# Notebook 05 — Evaluation
**EPPS–American Airlines Data Challenge — GROW 26.2**

Deep evaluation of model performance:
- Metrics per model
- Seasonal breakdown
- Risk score distribution
- Threshold analysis
- Top risky airport pairs

**Outputs:**
- `../outputs/results/model_metrics.txt`
- `../outputs/results/risky_pairs_ranked.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings, pickle
warnings.filterwarnings('ignore')

from sklearn.metrics import (classification_report, confusion_matrix,
                             average_precision_score, roc_auc_score,
                             precision_recall_curve, f1_score,
                             precision_score, recall_score)
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

PROC_PATH    = '../data/processed/'
MODELS_PATH  = '../outputs/models/'
FIGURES_PATH = '../outputs/figures/'
RESULTS_PATH = '../outputs/results/'
os.makedirs(RESULTS_PATH, exist_ok=True)
os.makedirs(FIGURES_PATH, exist_ok=True)

print('✅ Imports OK')

## 1 — Load Scored Data & Models

In [ ]:
df = pd.read_csv(f'{PROC_PATH}sequences_scored.csv', parse_dates=['Date'])

# Reload models
with open(f'{MODELS_PATH}baseline_lr.pkl','rb') as f:
    lr_model = pickle.load(f)

xgb_model = XGBClassifier()
xgb_model.load_model(f'{MODELS_PATH}xgboost_final.json')

print(f'Sequences loaded : {df.shape}')
print(f'Columns with scores: {[c for c in df.columns if "score" in c or "predicted" in c]}')

# Recreate test set (same split as notebook 04)
df_sorted = df.sort_values('Date').reset_index(drop=True)
split_idx  = int(len(df_sorted) * 0.70)
test_df    = df_sorted.iloc[split_idx:].copy()

ID_COLS  = ['airport_A','airport_B','Date','delayed_A','delayed_B',
            'is_risky','risk_score_xgb','risk_score_lr','predicted_risky_xgb']
OBJ_COLS = [c for c in df.columns if df[c].dtype == 'object']
FEAT_COLS = [c for c in df.columns if c not in ID_COLS + OBJ_COLS]

X_test = test_df[FEAT_COLS].fillna(0).astype(float)
y_test = test_df['is_risky'].astype(int)

y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_pred_xgb = xgb_model.predict(X_test)
y_prob_lr  = lr_model.predict_proba(X_test)[:, 1]
y_pred_lr  = lr_model.predict(X_test)

print(f'Test  set size : {len(X_test)}')
print(f'Risky in test  : {y_test.sum()} ({y_test.mean()*100:.1f}%)')

## 2 — Full Metrics Comparison

In [ ]:
results = {}
for name, y_pred, y_prob in [
    ('Logistic Regression', y_pred_lr,  y_prob_lr),
    ('XGBoost',             y_pred_xgb, y_prob_xgb)]:
    results[name] = {
        'PR-AUC'   : round(average_precision_score(y_test, y_prob), 4),
        'ROC-AUC'  : round(roc_auc_score(y_test, y_prob), 4),
        'F1'       : round(f1_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall'   : round(recall_score(y_test, y_pred), 4),
    }

results_df = pd.DataFrame(results).T
print('=== MODEL COMPARISON ===')
print(results_df.to_string())
print()

# Bar chart comparison
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
metrics = ['PR-AUC','ROC-AUC','F1','Precision','Recall']
colors_m = ['#3498db','#e74c3c']
for ax, metric in zip(axes, metrics):
    vals = [results[m][metric] for m in results]
    bars = ax.bar(list(results.keys()), vals, color=colors_m, edgecolor='white', width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
    ax.set_title(metric, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Model Performance Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}19_model_comparison.png', bbox_inches='tight')
plt.show()
print('💾 19_model_comparison.png')

## 3 — Risk Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, scores, title in zip(
    axes,
    [test_df['risk_score_lr'].values, test_df['risk_score_xgb'].values],
    ['Logistic Regression Risk Scores','XGBoost Risk Scores']):
    ax.hist(scores[y_test==0], bins=30, alpha=0.65, color='#2ecc71',
            label='Not Risky', density=True)
    ax.hist(scores[y_test==1], bins=30, alpha=0.65, color='#e74c3c',
            label='Risky', density=True)
    ax.axvline(0.5, color='black', ls='--', alpha=0.7, label='Threshold=0.5')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Risk Score')
    ax.set_ylabel('Density')
    ax.legend()
plt.suptitle('Risk Score Distribution — Risky vs Not Risky',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}20_risk_score_dist.png', bbox_inches='tight')
plt.show()
print('💾 20_risk_score_dist.png')

## 4 — Threshold Analysis (XGBoost)

In [ ]:
thresholds = np.arange(0.1, 0.91, 0.05)
f1s, precs, recs = [], [], []
for t in thresholds:
    pred = (y_prob_xgb >= t).astype(int)
    f1s.append(f1_score(y_test, pred, zero_division=0))
    precs.append(precision_score(y_test, pred, zero_division=0))
    recs.append(recall_score(y_test, pred))

best_t = thresholds[np.argmax(f1s)]
print(f'Best threshold by F1: {best_t:.2f} → F1={max(f1s):.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1s,   'o-', color='#9b59b6', lw=2, label='F1')
ax.plot(thresholds, precs, 's-', color='#3498db', lw=2, label='Precision')
ax.plot(thresholds, recs,  '^-', color='#e74c3c', lw=2, label='Recall')
ax.axvline(best_t, color='black', ls='--', alpha=0.7, label=f'Best threshold={best_t:.2f}')
ax.set_title('XGBoost — Threshold vs Precision / Recall / F1',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.legend()
ax.set_xlim([0.1, 0.9])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}21_threshold_analysis.png', bbox_inches='tight')
plt.show()
print('💾 21_threshold_analysis.png')

## 5 — Seasonal Evaluation

In [ ]:
season_col = 'season_A' if 'season_A' in test_df.columns else 'season'
test_df2 = test_df.copy()
test_df2['y_test']        = y_test.values
test_df2['y_prob_xgb']    = y_prob_xgb
test_df2['y_pred_xgb']    = y_pred_xgb

if season_col in test_df2.columns:
    seasons = test_df2[season_col].unique()
    print('=== SEASONAL BREAKDOWN ===')
    season_metrics = []
    for s in sorted(seasons):
        sub = test_df2[test_df2[season_col] == s]
        if len(sub) < 5 or sub['y_test'].sum() == 0:
            print(f'  {s}: insufficient data')
            continue
        pr  = average_precision_score(sub['y_test'], sub['y_prob_xgb'])
        roc = roc_auc_score(sub['y_test'], sub['y_prob_xgb'])
        f1  = f1_score(sub['y_test'], sub['y_pred_xgb'], zero_division=0)
        print(f'  {s:10s}: n={len(sub):4d} | Risky%={sub["y_test"].mean()*100:.0f}% | PR-AUC={pr:.3f} | ROC-AUC={roc:.3f} | F1={f1:.3f}')
        season_metrics.append({'season':s,'n':len(sub),'pr_auc':pr,'roc_auc':roc,'f1':f1})
    
    if season_metrics:
        sm_df = pd.DataFrame(season_metrics).set_index('season')
        fig, ax = plt.subplots(figsize=(10, 4))
        x = np.arange(len(sm_df))
        width = 0.25
        ax.bar(x-width, sm_df['pr_auc'],  width, label='PR-AUC',  color='#3498db', edgecolor='white')
        ax.bar(x,       sm_df['roc_auc'], width, label='ROC-AUC', color='#2ecc71', edgecolor='white')
        ax.bar(x+width, sm_df['f1'],      width, label='F1',      color='#e74c3c', edgecolor='white')
        ax.set_xticks(x)
        ax.set_xticklabels(sm_df.index)
        ax.set_ylim(0, 1.1)
        ax.set_title('XGBoost Performance by Season', fontweight='bold')
        ax.legend()
        plt.tight_layout()
        plt.savefig(f'{FIGURES_PATH}22_seasonal_evaluation.png', bbox_inches='tight')
        plt.show()
        print('💾 22_seasonal_evaluation.png')
else:
    print('Season column not found in test set')

## 6 — Top Risky Airport Pairs

In [ ]:
risky_pairs = (
    df.groupby(['airport_A','airport_B'])
    .agg(
        avg_risk_score  = ('risk_score_xgb', 'mean'),
        actual_risky_pct= ('is_risky',        'mean'),
        total_sequences = ('is_risky',        'count')
    )
    .reset_index()
    .sort_values('avg_risk_score', ascending=False)
)
risky_pairs['avg_risk_score']   = risky_pairs['avg_risk_score'].round(4)
risky_pairs['actual_risky_pct'] = (risky_pairs['actual_risky_pct'] * 100).round(1)

risky_pairs.to_csv(f'{RESULTS_PATH}risky_pairs_ranked.csv', index=False)
print('💾 risky_pairs_ranked.csv saved')
print()
print('=== TOP 20 RISKIEST AIRPORT PAIRS ===')
print(risky_pairs.head(20).to_string(index=False))

# Visualize top 15
top15 = risky_pairs.head(15).copy()
top15['pair'] = top15['airport_A'] + ' → DFW → ' + top15['airport_B']

fig, ax = plt.subplots(figsize=(12, 7))
cmap_vals = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(top15)))
bars = ax.barh(top15['pair'][::-1], top15['avg_risk_score'][::-1],
               color=cmap_vals, edgecolor='white')
for bar, (_, row) in zip(bars, top15[::-1].iterrows()):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f'Actual: {row["actual_risky_pct"]:.0f}%  n={int(row["total_sequences"])}',
            va='center', fontsize=8)
ax.set_title('Top 15 Riskiest Airport Pairs (A → DFW → B)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Average Risk Score (XGBoost)')
ax.set_xlim(0, 1.1)
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}23_risky_pairs_ranked.png', bbox_inches='tight')
plt.show()
print('💾 23_risky_pairs_ranked.png')

## 7 — Save Metrics Report

In [ ]:
report_lines = [
    'EPPS-American Airlines Data Challenge — GROW 26.2',
    'Model Evaluation Report',
    '='*55,
    '',
    f'Test set size : {len(y_test)}',
    f'Risky in test : {y_test.sum()} ({y_test.mean()*100:.1f}%)',
    '',
    '--- LOGISTIC REGRESSION ---',
    f'PR-AUC    : {average_precision_score(y_test, y_prob_lr):.4f}',
    f'ROC-AUC   : {roc_auc_score(y_test, y_prob_lr):.4f}',
    f'F1        : {f1_score(y_test, y_pred_lr, zero_division=0):.4f}',
    '',
    classification_report(y_test, y_pred_lr,
                          target_names=["Not Risky","Risky"]),
    '',
    '--- XGBOOST ---',
    f'PR-AUC    : {average_precision_score(y_test, y_prob_xgb):.4f}',
    f'ROC-AUC   : {roc_auc_score(y_test, y_prob_xgb):.4f}',
    f'F1        : {f1_score(y_test, y_pred_xgb, zero_division=0):.4f}',
    f'Best threshold (F1): {best_t:.2f}',
    '',
    classification_report(y_test, y_pred_xgb,
                          target_names=["Not Risky","Risky"]),
]

with open(f'{RESULTS_PATH}model_metrics.txt','w') as f:
    f.write('\n'.join(report_lines))
print('💾 model_metrics.txt saved')
print()
print('✅ Next → Notebook 06: SHAP Explainability')